# α-Weighted 与 Condition-Max 剪枝对照

同一个 Dense-Aug checkpoint 产生两组逐样本 CosFace Taylor 分数。一组来自干净训练图，另一组来自随机方块遮挡图。对每层分别归一化后，α-Weighted 计算加权平均，Condition-Max 对每个权重取两组分数的最大值。Magnitude 和未剪枝的 Dense-Aug 提供对照。

In [ ]:
from pathlib import Path
import sys

roots = [Path.cwd(), Path.cwd().parent, Path("/content/CS260D_Final_Project")]
repo = next(p for p in roots if (p / "src/occprune/prune_run.py").is_file())
sys.path.insert(0, str(repo / "src"))
from occprune.prune_run import run_pruning
from occprune.utils import load_json

### 查看已保存的结果

下面直接读取仓库内的逐条件结果，按打分种子展示 LFW 留出几何的平均准确率。确认数据准备完成后，再运行下方剪枝实验即可复现。

In [ ]:
import pandas as pd
from IPython.display import display

results_dir = repo / "results/pruning/one_shot_seed42"
if (results_dir / "metrics.csv").is_file():
    recorded = pd.read_csv(results_dir / "metrics.csv")
    regions = ["eyes", "mouth_nose", "upper_half", "lower_half"]
    subset = recorded[(recorded.dataset == "lfw") & recorded.condition.isin(regions)]
    table = 100 * subset.groupby(["method", "scoring_seed"]).accuracy.mean().unstack()
    display(table.round(2))
    print("Dense 和 Magnitude 共用 seed 42 的参照结果。")

先运行 notebook 01 准备 CASIA 和验证集缓存，从 README 的 Trained models 下载 seed 42 的 Dense-Aug。默认运行一个打分种子。正式 one-shot 对照使用 42、43、44 三个打分种子，同一份模型、α=0.5、每层剪 50%，每个种子共享 1,024 次评分反传。

In [ ]:
CHECKPOINT = repo / "checkpoints/dense_aug_seed42.pt"
SCORING_SEEDS = [42]  # 完整复核可改为 [42, 43, 44]
ALPHA, SPARSITY, BUDGET = 0.5, 0.5, 1024
FT_EPOCHS = 0
TEST_SETS = ()  # CASIA validation；正式设置冻结后设为 ("lfw",)
RUN = False  # 确认数据和权重就绪后改为 True
RESULTS = []
for seed in SCORING_SEEDS:
    tag = "_".join(TEST_SETS) if TEST_SETS else "validation"
    dest = repo / "results/pruning" / f"a{ALPHA}_s{SPARSITY}_b{BUDGET}_ft{FT_EPOCHS}_score{seed}_{tag}"
    if RUN:
        result = run_pruning(CHECKPOINT, dest, alpha=ALPHA, sparsity=SPARSITY,
                             budget=BUDGET, scoring_seed=seed, ft_epochs=FT_EPOCHS,
                             test_sets=TEST_SETS, eval_batch_size=64)
        RESULTS.append(result)
    elif (dest / "result.json").is_file():
        RESULTS.append(load_json(dest / "result.json"))
    else:
        print("确认设置后开启 RUN。输出目录：", dest)

每个新的输出目录记录参数计数、时间、原始条件分数、校准图片索引与方块、剪枝掩码及逐对相似度。相同的真实评分时间可归属于两个梯度方法，计算只进行一次。

In [ ]:
import pandas as pd
from IPython.display import display

rows = []
for r in RESULTS:
    for method, entry in r["methods"].items():
        stages = {"dense": entry} if method == "dense" else {
            k: entry[k] for k in ("one_shot", "fine_tuned") if k in entry}
        for stage, record in stages.items():
            for dataset, conditions in record["metrics"].items():
                row = {"seed": r["scoring_seed"], "method": method, "stage": stage, "dataset": dataset}
                row.update({k: 100*v["acc"] for k, v in conditions.items()})
                rows.append(row)
if rows:
    display(pd.DataFrame(rows).round(2))